# 🌍 AirSentinel Cameroun
## Notebook 06 — Analyse SHAP
**Responsable : PEURBAR RIMBAR Firmin — ISSEA**

### Objectif
Identifier les facteurs climatiques qui expliquent le mieux PM2.5
par zone climatique — Objectif OS3 du hackathon

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
import joblib, warnings
warnings.filterwarnings('ignore')

df       = pd.read_excel('../data/processed/dataset_final.xlsx')
df['date'] = pd.to_datetime(df['date'])
modele   = joblib.load('../models/meilleur_modele.pkl')
features = joblib.load('../models/features.pkl')
VAR_CIBLE = 'pm2_5_moyen'

df_model  = df[df[VAR_CIBLE].notna()].copy()
X_test    = df_model[df_model['date'] >= '2025-01-01'][features].fillna(0)
print(f'✅ Chargé — X_test : {X_test.shape}')

## Étape 1 — SHAP global

In [ ]:
X_shap    = X_test.sample(min(2000, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(modele)
vals_shap = explainer(X_shap)

# Graphique importance globale
shap.plots.bar(vals_shap, max_display=15, show=False)
plt.title('Importance des variables — AirSentinel Cameroun')
plt.tight_layout()
plt.savefig('../graphiques/shap_importance.png', dpi=300, bbox_inches='tight')
plt.show()

# Beeswarm
shap.plots.beeswarm(vals_shap, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../graphiques/shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Graphiques SHAP sauvegardés')

## Étape 2 — SHAP par zone climatique
**Innovation principale du projet**

In [ ]:
zones = {
    'Zone sahélienne':   ['Extrême-Nord', 'Nord'],
    'Zone équatoriale':  ['Centre', 'Est', 'Sud'],
    'Zone côtière':      ['Littoral', 'Sud-Ouest'],
    'Zone montagneuse':  ['Ouest', 'Nord-Ouest'],
    'Zone savane':       ['Adamaoua'],
}

print('Top 3 facteurs climatiques par zone :')
print('=' * 60)
resultats_zones = {}

for nom_zone, regions in zones.items():
    idx = df_model['region'].isin(regions)
    Xz  = df_model[idx & (df_model['date'] >= '2025-01-01')][features].fillna(0)
    Xz  = Xz.sample(min(500, len(Xz)), random_state=42) if len(Xz) > 50 else Xz

    if len(Xz) > 50:
        sv  = explainer(Xz)
        imp = pd.Series(abs(sv.values).mean(axis=0), index=features).sort_values(ascending=False)
        resultats_zones[nom_zone] = imp.head(3).to_dict()
        print(f'\n{nom_zone} :')
        for v, val in imp.head(3).items():
            print(f'  → {v:35s} : {val:.3f}')

pd.DataFrame(resultats_zones).T.to_excel('../data/processed/shap_par_zone.xlsx')
print('\n✅ SHAP par zone sauvegardé')

## Étape 3 — Interprétation narrative

In [ ]:
print('INTERPRÉTATION POUR LE RAPPORT')
print('=' * 60)
print()
print('Zone sahélienne (Extrême-Nord, Nord) :')
print('→ Harmattan dominant → dust + vent + température')
print()
print('Zone côtière (Littoral, Sud-Ouest) :')
print('→ Pollution industrielle + humidité + NO2')
print()
print('Zone équatoriale (Centre, Est, Sud) :')
print('→ Feux de brousse + CO + saison sèche')
print()
print('Zone montagneuse (Ouest, Nord-Ouest) :')
print('→ Altitude + température + précipitations')
print()
print('Ces résultats SHAP correspondent à l objectif OS3 du hackathon :')
print('Identifier les facteurs climatiques aggravants selon les zones')